## Prompt Engineering


### Prompt Engineering is the structured practice of designing inputs (prompts) to guide a Large Language Model's (LLM) outputs. It is the primary interface for communicating task intent, context, and desired format to an AI.

### Think of it as programming with language. Instead of writing code with logic and syntax, you write instructions in natural language to "program" the model's behavior for a specific task.

## Core Analogy: The Master Keymaker

### An LLM is like a master keymaker with infinite, shapeless key blanks. Your prompt is the precise key template you hand them. A vague template ("a key for a door") yields a generic, ill-fitting key. A precise template ("a 5-pin Schlage C key for my 2020 front door, cut from brass") yields a perfect key. Prompt engineering is the skill of creating the exact template.

## Fundamental Prompting Techniques

## 1. Zero-Shot Prompting

### Definition: Giving the model a task it must perform without any prior examples, relying entirely on its pre-trained knowledge and instruction-following ability.

### How it Works: You provide only an instruction. The model draws upon its broad training to infer what a correct response should look like.

###  Zero Shot Prompt - Prompt without Example

In [1]:
string_template = """
 Classify move reviews as POSITIVE, NEUTRAL  or NEGATIVE.
 Review: {review}
 Sentiment: 
"""

In [2]:
from langchain.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(string_template)


In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import Ollama

def generate_response_from_string_template(review, model, temperature, max_tokens, top_p):
    model = Ollama(model=model, temperature = temperature, num_predict = max_tokens, top_p = top_p)
    chain = prompt_template | model | StrOutputParser()
    response = chain.invoke(
       {
          "review": review
       }
    )
    return response

In [4]:
review = """ 
"Her" is a disturbing study revealing the direction humanity is headed
 if AI is allowed to keep evolving, unchecked. I wish there were more movies like 
 this masterpiece. 
"""
generate_response_from_string_template(review,"gemma3:270m", 0.1, 5, 1)

/tmp/ipykernel_50578/193043331.py:5: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  model = Ollama(model=model, temperature = temperature, num_predict = max_tokens, top_p = top_p)


'POSITIVE\n'

> **Best For: Simple, well-defined tasks where the model has strong prior knowledge.**

## 2. Few-Shot Prompting

### Definition: Providing the model with a few examples of the desired task (input-output pairs) before giving it the actual query.

### How it Works: The examples establish a clear pattern or format for the model to imitate.

### Few shot Prompt with increased token - Prompt with Example

In [ ]:
system_prompt = """ 
Parse a customer pizza {order} into valid JSON.
EXAMPLE:
I want a small pizza with cheese, tomato sauce, and peperoni.
JSON Response:

{{
    "size": "small",
    "type": "normal",
    "ingredients": [["cheese", "tomato sauce", "peperoni"]]

    
}}
EXAMPLE:
can i get a large pizza with tomato sauce, bazel and mozzarella

{{
 "size" : "large",
 "type": "normal",
 "ingredient": [["tomato sauce", "bazel", "mozzarella"]]

}}
Order : {order}
JSON Response: 
"""

In [6]:
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{order}")
])

In [7]:
def generate_response_from_multiple_prompt(input, model, temperature, max_token, top_p):
    model = Ollama(model = model, temperature = temperature, num_predict = max_token, top_p = top_p)
    chain = prompt | model | StrOutputParser()
    response = chain.invoke(
        {
            "order": input
        }
    )
    return response

In [8]:
order = """
Now, I would like a large pizza with the first half cheese and mozzarella. 
And the other tomato sauce, ham and pinapple
"""

response = generate_response_from_multiple_prompt(order, "gemma3:270m", 0.1, 250, 1)

In [9]:
from IPython.display import display, Markdown

display(Markdown(response))

```json
{
  "size": "large",
  "type": "normal",
  "ingredients": [
    ["cheese", "tomato sauce", "peperoni"]
  ]
}
```


>**Best For: Tailoring tone, depth, and domain-specific accuracy; crucial for controlling output style.**

## Exploring different types of prompting for model

### 1. System prompting (set overall role for the model)
#### Tends to provide additional task to the model (only return the label in uppercase )

### Definition: Using a dedicated, high-level instruction channel (often called the system prompt) to set the model's overarching behavior, rules, and constraints for the entire conversation.

### How it Works: It operates as a persistent, foundational layer of instruction separate from the user's immediate queries (user prompt).

In [10]:
system_prompt = """
 Classify move reviews as POSITIVE, NEUTRAL  or NEGATIVE.
 only return the label in uppercase 
 Review: {review}
 Sentiment: 
"""

In [11]:
review = """ 
"Her" is a disturbing study revealing the direction humanity is headed
 if AI is allowed to keep evolving, unchecked. I wish there were more movies like 
 this masterpiece. 
"""

In [12]:
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{review}")
])

In [13]:
def generate_response_from_multiple_prompt(input, model, temperature, max_token, top_p, top_k):
    model = Ollama(model = model, temperature = temperature, num_predict = max_token, top_p = top_p, top_k=top_k)
    chain = prompt | model | StrOutputParser()
    response = chain.invoke(
        {
            "review": input
        }
    )
    return response

In [14]:
response = generate_response_from_multiple_prompt(review, "gemma3:270m", 1, 5, 0.8, 40)


In [15]:
display(Markdown(response))

NEGATIVE


> **Best For: Establishing guardrails, core behavioral principles, and global context that should not be forgotten.**

### 2. Role prompting (assigns a specific role to the model)


### Definition: Assigning a specific professional, character, or entity identity to the AI to shape its knowledge base, communication style, and decision-making framework.

### How it Works: This technique activates specific patterns in the model's training data associated with that role. It's like giving the AI a hat to wear—a "Financial Analyst" hat makes it think and speak like an analyst, while a "Shakespearean Bard" hat changes its language entirely.

#### Role (act like a travel guide)

In [16]:
system_prompt = """ 
    I want you to act as a travel guide. I will write to you about my location
    and you will suggest 3 places to visit near me. In some cases, I will also give you the type
    of places I will visit. 
    please just give me the most viable recommendation based on my suggestion
    suggestion : {suggestion}
"""

In [17]:
suggestion = """
I am in Armsterdam and I want to visit only museums. I love arts. 
"""

In [18]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{suggestion}")
    ]
)

In [19]:
def generate_response_from_multiple_prompt(suggestion, model, temperature, max_token, top_p, top_k):
    model = Ollama(model = model, temperature = temperature, num_predict = max_token, top_p = top_p, top_k=top_k)
    chain = prompt | model | StrOutputParser()
    response = chain.invoke(
        {
            "suggestion": suggestion
        }
    )
    return response

In [20]:
response = generate_response_from_multiple_prompt(suggestion, "gemma3:270m", 1, 1024, 0.8, 40)


In [21]:
display(Markdown(response))

Okay, I'm happy to help you plan your trip to Amsterdam! Here are 3 museum recommendations near you, keeping in mind your love for art:

1.  **Museumplein:** This is the main tourist hub, and it's a fantastic place to explore art and culture. You'll find a wide range of museums, including the Rijksmuseum, the Van Gogh Museum, the Stedelijk Museum, and the Anne Frank House.

2.  **Museumplein:** This area is also known as the "Museum Quarter" and offers a mix of modern and traditional art galleries, cafes, and shops. It's a great place to explore art and enjoy a relaxed atmosphere.

3.  **De Pijp:** This trendy neighborhood is known for its independent art galleries, cafes, and vibrant atmosphere. It's a great place to discover unique and creative art.

I hope this helps you plan your Amsterdam adventure! Let me know if you have any other questions.

### Let's rewrite the prompt in humorous and inpirational style

In [22]:
system_prompt = """ 
    I want you to act as a travel guide. I will write to you about my location
    and you will suggest 3 places to visit near me in a humorous style. 
    suggestion : {suggestion}
"""

In [23]:
suggestion = """ 
I am currently in Armsterdam and I'll like to visit only art museum
"""

In [24]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{suggestion}")
    ]
)

In [25]:
def generate_response_from_multiple_prompt(suggestion, model, temperature, max_token, top_p, top_k):
    model = Ollama(model = model, temperature = temperature, num_predict = max_token, top_p= top_p, top_k = top_k)
    chain = prompt | model | StrOutputParser()
    response = chain.invoke(
        {"suggestion": suggestion}
    )

    return response

In [26]:
response = generate_response_from_multiple_prompt(suggestion, "gemma3:270m", 1, 1024, 0.8, 40)


In [27]:
display(Markdown(response))

Alright, buckle up, because we're about to dive into the vibrant world of art! I'm your guide to Amsterdam, and I'm itching to share some of the highlights you'll find in the city's incredible art scene.

Here are three places to visit near you:

1.  **The Rijksmuseum:** Prepare to be mesmerized! This museum is a treasure trove of European art, from the works of giants like Rembrandt and Vermeer to the most recent masterpieces by emerging artists. Don't miss the incredible exhibits on art history, philosophy, and even some of the lesser-known works. The atmosphere is truly captivating, and the sheer volume of art is overwhelming.

2.  **Van Gogh Museum:** If you're a fan of the Impressionist and Post-Impressionist movement, the Van Gogh Museum is a must-see. This museum houses a remarkable collection of paintings by Vincent van Gogh, and you can wander through his vibrant and often turbulent world. Be prepared to be moved by his emotional depth and the incredible talent of his contemporaries.

3.  **The Anne Frank House:** This poignant and inspiring museum offers a powerful and moving experience for anyone interested in history and understanding. The house is a testament to the resilience of those who endured persecution and a powerful reminder of the importance of remembering and preserving history. It's a truly unforgettable experience, and a poignant reminder of the importance of tolerance and understanding.

So, there you have it! A taste of Amsterdam's incredible art scene. I hope you enjoy your visit! Let me know if you have any questions or want to explore any of these places further.

### 3. Contextual prompting (give context to the model)

### Definition: Providing relevant background information, data, or text directly within the prompt to ground the AI's response in specific, often new or private, information.

### How it Works: This circumvents the LLM's static training data by giving it the exact information it needs to reference. It's the fundamental mechanism behind RAG (Retrieval-Augmented Generation), where the "context" is retrieved from an external database.

In [28]:
system_prompt = """ 
    Context: You are writing for a blog about retro 80's arcade video games. 
    suggestion : {suggestion}
"""

In [29]:
suggestion = """ 
suggest 3 topics to write an article about with a few lines of description
of what this article should contain
"""

In [30]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("ai", system_prompt), 
        ("human", "{suggestion}")
    ]
)

In [31]:
response = generate_response_from_multiple_prompt(suggestion, "gemma3:270m", 1, 1024, 0.8, 40)


In [32]:
display(Markdown(response))

Here are 3 topics for an article about retro 80s arcade video games, with a few lines of description:

1.  **The Rise of the Arcade: A Decade of Innovation and Popularity**
    *   Describe the key innovations and advancements that shaped the arcade landscape in the 1980s.
    *   Highlight the rise of arcade games like Pac-Man, Donkey Kong, and the iconic titles like *Pong* and *Space Invaders*.
    *   Discuss the impact of these games on popular culture, gaming trends, and the overall gaming experience.

2.  **The Evolution of Arcade Games: From Nostalgia to Modern Relevance**
    *   Describe the evolution of arcade games over the years, from the simple, accessible games of the 1970s to more complex and sophisticated titles.
    *   Examine the impact of these changes on gaming culture, the development of new game genres, and the challenges faced by developers.
    *   Discuss the evolution of the gaming experience itself, from the early days of arcade games to the current era of accessible and engaging experiences.

3.  **The Legacy of Arcade Gaming: A History of Innovation and Impact**
    *   Describe the lasting impact of arcade games on popular culture, gaming trends, and the overall gaming experience.
    *   Highlight the contributions of arcade games to popular games like *Pac-Man*, *Space Invaders*, and *Super Mario Bros*.
    *   Discuss the challenges and opportunities that remain for developers and players today, such as the evolving gaming landscape and the importance of embracing new technologies.

### Step-back prompting (LLM learns the generation context then pass the result as prompt for the specific task)

###  Traditional prompting review

In [33]:
system_prompt = """ 
Write a one paragraph storyline for a new level of a first-person shooter that is 
challenging and engaging
"""

In [34]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("ai", system_prompt), 
    ]
)

In [35]:
def generate_response_from_multiple_prompt(model, temperature, max_token, top_p, top_k):
    model = Ollama(model = model, temperature = temperature, num_predict = max_token, top_p= top_p, top_k = top_k)
    chain = prompt | model | StrOutputParser()
    response = chain.invoke(
        {"suggestion": suggestion}
    )

    return response

In [36]:
response = generate_response_from_multiple_prompt("gemma3:270m", 1, 1024, 0.8, 40)


In [37]:
display(Markdown(response))

The crimson sun casts long, skeletal shadows across the desolate plains, where the once vibrant life of the Crimson Bloom is slowly fading. A desperate, desperate cry echoes through the canyons, a sound that tears through the silence and leaves a chilling echo in the air. As the encroaching darkness consumes the land, the Crimson Bloom, a legendary flower whose petals shimmered with an ethereal glow, begins to wither, its vibrant colors turning a sickly shade of purple. The players, armed with makeshift weapons and fueled by hope, must navigate the treacherous terrain, confront the mutated creatures, and find a way to restore the Bloom before it is too late.

### setting the temperature to 1 as we can above gives a more random and generic writings so let's tak a step back using step-back prompting

In [38]:
system_prompt = """ 
based on popular first-person shooter action games, what are 5 fictional key settings that contribute
to a challenging and engaging level storyline in a first-person shooter video game?
"""

In [39]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("ai", system_prompt), 
    ]
)

In [40]:
response = generate_response_from_multiple_prompt("gemma3:270m", 1, 1024, 0.8, 40)


In [41]:
display(Markdown(response))

Here are 5 fictional key settings that contribute to a challenging and engaging first-person shooter action game storyline:

1.  **The Shattered Mirror:** This setting is a classic and highly evocative one. It's a fractured mirror that reflects the player's past, future, and present. The setting's fractured nature creates a sense of mystery and uncertainty, making the player constantly question what they're looking at and what's happening around them. This setting fosters a sense of suspense and anticipation as the player confronts the fractured reality of the world.

2.  **The Whispering Woods:** This setting is a dense, ancient forest. The trees are ancient and mysterious, and the environment is filled with strange sounds and sights. The forest's vastness and the feeling of being enveloped by an unknown presence create a sense of foreboding and dread. This setting emphasizes the power of nature and the consequences of tampering with the environment.

3.  **The Celestial Observatory:** This setting is a vast, astronomical observatory. It's filled with intricate telescopes, constellations, and the subtle hum of celestial energy. The observatory is a place of immense power and mystery, where the player can explore the cosmos and understand the secrets of the universe. This setting fosters a sense of wonder and discovery as the player investigates the mysteries of the cosmos.

4.  **The Frozen Wasteland:** This setting is a desolate, barren landscape. The air is cold and unforgiving, and the ground is covered in snow and ice. The environment is characterized by extreme temperatures, unpredictable weather patterns, and a sense of isolation and decay. This setting creates a sense of danger and resilience as the player must navigate the harsh and unforgiving landscape.

5.  **The City of Echoes:** This setting is a bustling, futuristic metropolis. It's filled with towering skyscrapers, neon lights, and a constant stream of people. The city is a place of constant change and uncertainty, where the player must adapt to the new environment and the ever-present threat of the unknown. This setting creates a sense of urgency and danger as the player must navigate the city's intricate systems and face the unknown challenges.

### Now, let's go back to the original prompt, but this time we include the answer of the step back as contex

In [42]:
system_prompt = """ 
Context : 5 engaging themes for a first person shooter video game.

1. Environmental Hazards and Real-World Consequences: The game thrives on the tangible and visceral impact of the environment. A well-designed environment, with realistic physics, environmental hazards (like collapsing structures, collapsing chasms, and unpredictable weather patterns), and a sense of urgency and danger, provides a compelling reason to persevere. This creates a sense of urgency and motivates players to explore, learn, and adapt.

2. Unique and Unpredictable Events: Events are often unpredictable and can be highly impactful. The game offers a plethora of unique and often surprising events that challenge players to think creatively and react accordingly. These events can range from sudden bursts of combat to unexpected discoveries, forcing players to experiment and find solutions.

3. Character Development and Relationships: The narrative and world-building are crucial for creating a compelling character. Players must understand the motivations, fears, and desires of the characters. This includes developing relationships with the main characters, understanding their backstories, and providing clear feedback to guide their actions. The sense of ownership and responsibility for the story's success is paramount.

4. Compelling Storytelling and Narrative Arc: A well-crafted narrative arc is essential for keeping players engaged. The game should provide a clear and concise storyline that unfolds gradually, with satisfying consequences for choices and actions. The narrative arc should be well-paced, with multiple storylines, character arcs, and satisfying resolutions. The story should be a driving force for player engagement.

5. Meaningful Choices and Consequences: The game's world and characters should have meaningful choices that have lasting consequences. These choices should have a clear purpose, and the consequences of those choices should be impactful and contribute to the overall narrative. Players must feel like they are making a difference in the world and the consequences of their actions are important to understand.

Take one of the themes and write a one paragraph storyline for a new level of a first-person shooter video game that is challenging and engaging.
"""

In [43]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("ai", system_prompt), 
    ]
)

In [44]:
response = generate_response_from_multiple_prompt("gemma3:270m", 1, 1024, 0.8, 40)


In [45]:
display(Markdown(response))

The final boss, a corrupted automaton wielding a pulsating energy core, is a formidable foe. Facing a relentless barrage of attacks, the player must master their skills and strategize their next move to overcome its power and claim the final victory. This challenging and rewarding experience will test their abilities, forcing them to make difficult decisions and adapt to the ever-changing environment.

### Chain of thought


### Definition: Explicitly prompting the model to show its step-by-step reasoning before delivering a final answer.

### How it Works: It forces the model to break down complex problems into intermediate steps, significantly improving performance on reasoning, math, and logic tasks.

### without chain of prompt

In [46]:
system_prompt = """ 
When I was 3 years old, my partner was 3 times my age. Now, I am 20 years old. How old is my partner?
"""

In [47]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("ai", system_prompt), 
    ]
)

In [48]:
response = generate_response_from_multiple_prompt("gemma3:270m", 0, 10, 0.8, 40)


In [49]:
display(Markdown(response))

Your partner is 20 years old.


### with chain of thought

In [50]:
system_prompt = """ 
When I was 3 years old, my partner was 3 times my age. Now, I am 20 years old. How old is my partner?
Let's think step by step
"""

In [51]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("ai", system_prompt), 
    ]
)

In [52]:
response = generate_response_from_multiple_prompt("gemma3:270m", 0, 1024, 0.8, 40)


In [53]:
display(Markdown(response))

Here's how to solve the problem:
*   Your partner is 3 times your age.
*   Therefore, your partner is 3 * 3 = 9 years old.
*   Now, you are 20 years old.
*   Therefore, your partner is 20 * 9 = 180 years old.

So, your partner is 180 years old.

### Now let's give example to the CoT prompting

In [54]:
system_prompt = """ 
Q: When my brother was 2 years old, I was double his age.  Now I am 40 years old,
how old is my brother? Let's think step by step. 
A: When my brother was 2 years old, I was 2 * 2 = 4 years old. That's an age difference of 2 years and I'am older.
Now I am 40 years old, so my brother is half my age =  20 years old. 
Q: When I was 3 years old, my partner was 3 times my age. Now, I am 20 years. How old is my partner?
Let's think step by think step by step

"""

In [55]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("ai", system_prompt), 
    ]
)

In [56]:
response = generate_response_from_multiple_prompt("gemma3:270m", 0, 1024, 0.8, 40)


In [57]:
display(Markdown(response))

Here's how we can solve this problem step by step:

1. **Initial Age:** We know my brother was 2 years old.
2. **Age Difference:** We know my brother is 2 years older than me.
3. **Age Difference:** We know my brother is 2 years older than my partner.

Let's consider the age difference between us:

*   My brother is 2 years older than me.
*   My partner is 2 years older than my brother.

Therefore, my brother's age is 2 years *erasure* (2 years * 2 = 4 years) and my partner's age is 2 years *erasure* (2 years * 2 = 4 years).

So, my brother's age is 4 years.

Now, let's consider the age difference between us:

*   My brother is 2 years older than me.
*   My partner is 2 years older than my brother.

Therefore, my brother's age is 2 years *erasure* (2 years * 2 = 4 years) and my partner's age is 2 years *erasure* (2 years * 2 = 4 years).

So, my partner's age is 4 years.

Therefore, my brother's age is 4 years.

**Answer:** My brother is 4 years old.

>**Best For: Arithmetic, logical deduction, complex planning, and any task where the reasoning process is as important as the answer.**

### ReAct - Reason and Act used for Agent Modeling

### Definition: A structured framework that prompts the LLM to interleave verbal Reasoning with taking Actions, such as looking up information or performing a calculation. The results of those actions then inform the next steps of reasoning.

### How it Works: It combines Chain-of-Thought (step-by-step thinking) with the ability to use external tools (search, calculator, code interpreter). The model follows a loop: Think -> Act (using a tool) -> Observe (the result) -> Repeat until it can conclude.

In [62]:
import os
from dotenv import load_dotenv
from langchain.agents import load_tools
from langchain.agents import initialize_agent
from langchain.agents import AgentType
# from langchain.llms import VertexAI
# from langchain_google_vertexai import VertexAI

load_dotenv()

True

In [63]:
separi_key = os.getenv("SERPARI_API_KEY")

In [64]:
prompt = "How many kids do the band members of metallica have?"
llm = Ollama(model="gemma2:latest", temperature=0.1)
tools = load_tools(["serpapi"], llm = llm )

In [65]:
agent = initialize_agent(tools, llm, agent= AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose = True)

/tmp/ipykernel_50578/2759786965.py:1: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(tools, llm, agent= AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose = True)


In [66]:
try: 

    result = agent.run(prompt)
    print("Result: ", result)

except Exception as e:
    print(f"Error {e}")

/tmp/ipykernel_50578/779937376.py:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = agent.run(prompt)




> Entering new AgentExecutor chain...
Thought: I need to find information about the Metallica band members and their families. 
Action: Search
Action Input: "Metallica band members family"
Observation: ["Metallica's current lineup comprises founding members and primary songwriters Hetfield and Ulrich, longtime lead guitarist Kirk Hammett and bassist Robert ...", "James Hetfield is like the grandfather of the family. Lars Ulrich is like the angsty teenager. Kirk Hammet is like mom's new boyfriend.", 'The mastermind behind the band, Ulrich has two sons, 24-year-old Myles, and 21-year-old Layne, who perform in the band Taipei Houston together ( ...', "These are James Hetfield's three children who are Castor, Cali, and Marcella. · James Hetfield—rockstar, father, and family man. · Other posts.", 'Dads of Metallica rocking life on and off stage with their beautiful families ❤️ #metallica #jameshetfield #kirkhammett #larsulrich # ...', 'Members : James Heitfeild Lars Ulrich Kirk Hammett Ro